In [ ]:
import argparse
import sys
from pathlib import Path
import os
import pandas as pd

# In a notebook, abspath('') is already this notebook's directory
# (architecture/preprocessing) — no .parent, unlike __file__ in the script.
_HERE = Path(os.path.abspath('')).resolve()
ARCH_DIR = _HERE.parent
REPO_ROOT = ARCH_DIR.parent
# fed_stroke (frozen schema) + repo root (registry_alignement) + the
# registry_alignement dir itself (so `from mappings import ...` inside
# build_gva_summary_table resolves — same shim as build_gva_first_values_table).
for _p in (ARCH_DIR, REPO_ROOT, REPO_ROOT / "registry_alignement"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


from registry_alignement import build_gva_summary_table
from registry_alignement.geneva_preprocessing import utils
from fed_stroke.schema import FEATURE_COLS, FEATURE_UNITS, TARGET_COL  # noqa: E402

SCHEMA_VERSION = "frozen-v1"   # bump on any FEATURE_COLS/FEATURE_UNITS/TARGET_COL change
PROVENANCE = "real-frozen-schema"


def build_frozen_gva_table(registry_xlsx: Path, ehr_dir: Path) -> pd.DataFrame:
    """Registry + EHR → ONE tidy per-admission table in the frozen schema.

    Steps to populate (reuse, don't re-derive):
    - cohort: registry_alignement.build_gva_summary_table.preprocess
      (exact-duplicate rows dropped, 'Type of event' == 'Ischemic stroke');
    - outcome: the OPSUM 3M Death / 3M mRS reconciliation already coded there;
    - case_admission_id: preprocessing.prepare_geneva_halves.build_case_admission_id
      (== the loader's patient_id + '_' + EDS-last-4 derivation, task.py);
    - EHR features: registry_alignement.build_gva_first_values_table extraction,
      joined on case_admission_id;
    - unit-convert into FEATURE_UNITS (mappings.UNIT_CONVERSIONS), rename to the
      frozen names (mappings.GVA_TO_FROZEN), then mappings.validate_frozen_columns.

    Input:
        registry_xlsx: the Geneva stroke-registry export (.xlsx).
        ehr_dir: the EHR extraction directory (patientvalue + lab CSVs).
    Output:
        DataFrame, one row per case_admission_id, columns EXACTLY
        ['case_admission_id', *FEATURE_COLS, TARGET_COL]:
        - case_admission_id: str, '<patient_id>_<eds_last_4>';
        - features: float, IN FEATURE_UNITS, missing = NaN (no sentinel),
          out-of-range values resolved HERE (data-quality error, not a
          privacy question);
        - target: int in {0, 1}; rows with underivable outcome dropped
          (count reported via the summary).
    """
    # preprocess registry
    df = pd.read_excel(registry_xlsx)
    df, n_raw, n_filtered = build_gva_summary_table.build_cohort(df)
    # derive case_admission_id
    df['case_admission_id'] = utils.create_registry_case_identification_column(df)
    df = build_gva_summary_table.preprocess_outcome(df)
    df = build_gva_summary_table.preprocess_features(df)

    return df


In [ ]:
registry_xlsx_path = '/Users/jk/stroke_datasets/stroke_registry_post_hoc_modified.xlsx'
ehr_dir_path = ''

In [ ]:
df = build_frozen_gva_table(Path(registry_xlsx_path), Path(ehr_dir_path))

In [ ]:
df.shape

In [ ]:
from registry_alignement import build_gva_first_values_table
df["admission_date"] = build_gva_first_values_table.parse_yyyymmdd(df["Arrival at hospital"])

In [ ]:
from registry_alignement.build_gva_first_values_table import (
    load_concat_csvs,
    ,
    extract_pv_vital_first_values,
    extract_pv_lab_first_values,
    extract_lab_dosage_first_values,
)
print(f"[cohort] n_patients={len(cohort)}")

print(f"[load]   PV files ({args.vitals_prefix}*.csv) from {ehr_dir}")
vitals_df = load_concat_csvs(ehr_dir, args.vitals_prefix)
vitals_df["case_admission_id"] = utils.create_ehr_case_identification_column(vitals_df)
print(f"[load]   PV rows={len(vitals_df)}")

print(f"[load]   lab files ({args.lab_prefix}*.csv) from {ehr_dir}")
lab_df = load_concat_csvs(ehr_dir, args.lab_prefix)
lab_df["case_admission_id"] = utils.create_ehr_case_identification_column(lab_df)
print(f"[load]   lab rows={len(lab_df)}")

per_var: dict[str, pd.DataFrame] = {}
per_var.update(extract_pv_vital_first_values(vitals_df, cohort))
per_var.update(extract_pv_lab_first_values(vitals_df, cohort))
per_var.update(extract_lab_dosage_first_values(lab_df, cohort))

for var, frame in per_var.items():
    n_with = int(frame["case_admission_id"].nunique()) if not frame.empty else 0
    print(f"[stat]   {var:<22s} patients_with_first_value={n_with}")
